# Scratch Patch Heavy Experiments

Notebook strictement **from-scratch** : aucun modele pretrained, aucun transfert learning.

Objectif : pousser la meilleure piste observee, le patch-based learning, avec deux CNN plus forts :
- SE-ResNet34 patch
- WideResNet28-10 patch

Chaque modele sauvegarde les probabilites par fold, les CSV par fold, le CSV 5-fold, puis un ensemble des modeles disponibles.


## 1. Setup


In [1]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Root  :', ROOT)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Root  : /home/onyxia/work/tilda-texture-classification
Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Config


In [2]:
N_SPLITS = 5
NUM_CLASSES = 8
NUM_WORKERS = 0

PATCH_RESIZE = (512, 768)
PATCH_SIZE = 384
BATCH_SIZE = 32

EPOCHS = 160
PATIENCE = 35
LR = 0.08
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05

RUN_MODELS = [
    'se_resnet34_patch_heavy',
    'wide_resnet28_10_patch',
]

START_FOLD = {
    'se_resnet34_patch_heavy': 1,
    'wide_resnet28_10_patch': 1,
}

print('Config OK')


Config OK


## 3. Data


In [3]:
def read_train_csv(path):
    try:
        df = pd.read_csv(path, sep=';')
        if len(df.columns) == 1:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df['id'] = df['id'].astype(int)
    df['label'] = df['label'].astype(int)
    return df


def resolve_image_path(folder, image_id):
    stem = str(int(image_id))
    for ext in ['.tif', '.tiff', '.png', '.jpg', '.jpeg']:
        p = folder / f'{stem}{ext}'
        if p.exists():
            return p
    matches = sorted(folder.glob(f'{stem}.*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Image introuvable: {image_id}')

train_dir = DATA_DIR / 'train'
test_dir = DATA_DIR / 'test'
df = read_train_csv(DATA_DIR / 'train.csv')
df['path'] = df['id'].map(lambda x: resolve_image_path(train_dir, x))

test_paths = sorted(test_dir.glob('*.*'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({'id': [int(p.stem) for p in test_paths], 'path': test_paths})

print(df['label'].value_counts().sort_index())
print('Train images:', len(df))
print('Test images :', len(test_df))
print('Example:', Image.open(df.iloc[0]['path']).mode, Image.open(df.iloc[0]['path']).size)


label
0    300
1    300
2    262
3    300
4    300
5    299
6    300
7    300
Name: count, dtype: int64
Train images: 2361
Test images : 789
Example: L (768, 512)


## 4. Patch transforms


In [4]:
train_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.RandomCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.05, scale=(0.01, 0.03), ratio=(0.3, 3.3)),
])

val_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.CenterCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

test_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

class TildaDataset(Dataset):
    def __init__(self, frame, transform=None, has_labels=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label']) if self.has_labels else -1
        return image, label, int(row['id'])


def make_loader(frame, transform, batch_size=BATCH_SIZE, shuffle=False, has_labels=True):
    ds = TildaDataset(frame, transform=transform, has_labels=has_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=torch.cuda.is_available())


## 5. Models from scratch


In [5]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)
        return x * w

class SEBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.se = SEBlock(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                                          nn.BatchNorm2d(planes))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.relu(out + self.shortcut(x), inplace=True)

class SEResNet(nn.Module):
    def __init__(self, layers, num_classes=8):
        super().__init__()
        self.in_planes = 64
        self.stem = nn.Sequential(nn.Conv2d(1, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64),
                                  nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        self.layer1 = self._make_layer(64, layers[0], 1)
        self.layer2 = self._make_layer(128, layers[1], 2)
        self.layer3 = self._make_layer(256, layers[2], 2)
        self.layer4 = self._make_layer(512, layers[3], 2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)
        self.apply(init_weights)
    def _make_layer(self, planes, blocks, stride):
        layers = [SEBasicBlock(self.in_planes, planes, stride)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(SEBasicBlock(self.in_planes, planes, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.fc(self.pool(x).flatten(1))

class WideBasic(nn.Module):
    def __init__(self, in_planes, planes, dropout_rate, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)
    def forward(self, x):
        out = self.conv1(F.relu(self.bn1(x), inplace=True))
        out = self.dropout(out)
        out = self.conv2(F.relu(self.bn2(out), inplace=True))
        return out + self.shortcut(x)

class WideResNet(nn.Module):
    def __init__(self, depth=28, widen_factor=10, dropout_rate=0.1, num_classes=8):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n = (depth - 4) // 6
        widths = [16, 16*widen_factor, 32*widen_factor, 64*widen_factor]
        self.in_planes = widths[0]
        self.conv1 = nn.Conv2d(1, widths[0], 3, 1, 1, bias=False)
        self.block1 = self._wide_layer(widths[1], n, dropout_rate, 1)
        self.block2 = self._wide_layer(widths[2], n, dropout_rate, 2)
        self.block3 = self._wide_layer(widths[3], n, dropout_rate, 2)
        self.bn = nn.BatchNorm2d(widths[3])
        self.fc = nn.Linear(widths[3], num_classes)
        self.apply(init_weights)
    def _wide_layer(self, planes, num_blocks, dropout_rate, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(WideBasic(self.in_planes, planes, dropout_rate, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.conv1(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = F.relu(self.bn(x), inplace=True)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)

def build_model(name):
    if name == 'se_resnet34_patch_heavy':
        return SEResNet([3, 4, 6, 3], NUM_CLASSES)
    if name == 'wide_resnet28_10_patch':
        return WideResNet(depth=28, widen_factor=10, dropout_rate=0.1, num_classes=NUM_CLASSES)
    raise ValueError(name)

for name in RUN_MODELS:
    m = build_model(name)
    print(name, f'{sum(p.numel() for p in m.parameters())/1e6:.1f}M params')
    del m


se_resnet34_patch_heavy 21.4M params
wide_resnet28_10_patch 36.5M params


## 6. Train / predict


In [6]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, n = 0.0, 0, 0
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)
            if is_train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += labels.size(0)
    return correct / n, total_loss / n


def fit_model(model, fold_name, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                                weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
    best_acc, best_epoch = -1.0, 0
    best_path = CHECKPOINT_DIR / f'best_{fold_name}.pt'
    history = []
    start = time.time()
    for epoch in range(1, EPOCHS + 1):
        tr_acc, tr_loss = run_epoch(model, train_loader, criterion, optimizer)
        va_acc, va_loss = run_epoch(model, val_loader, criterion, None)
        scheduler.step()
        history.append({'epoch': epoch, 'train_acc': tr_acc, 'train_loss': tr_loss,
                        'val_acc': va_acc, 'val_loss': va_loss, 'lr': scheduler.get_last_lr()[0]})
        if va_acc > best_acc:
            best_acc, best_epoch = va_acc, epoch
            torch.save({'model_state_dict': model.state_dict(), 'best_acc': best_acc}, best_path)
        print(f'{fold_name} | ep {epoch:03d} | train {tr_acc:.4f}/{tr_loss:.4f} | val {va_acc:.4f}/{va_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')
        if epoch - best_epoch >= PATIENCE:
            print(f'Early stop: no improvement for {PATIENCE} epochs')
            break
    return pd.DataFrame(history), best_path, best_acc, best_epoch, (time.time() - start)/60


def crop_grid(images, crop_size=384):
    _, _, h, w = images.shape
    tops = [0, max((h - crop_size)//2, 0), max(h - crop_size, 0)]
    lefts = [0, max((w - crop_size)//2, 0), max(w - crop_size, 0)]
    return [images[:, :, t:t+crop_size, l:l+crop_size] for t in tops for l in lefts]

@torch.no_grad()
def predict_patch_tta(model, loader):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        prob_list = []
        for crop in crop_grid(images, PATCH_SIZE):
            variants = [crop, torch.flip(crop, dims=[-1]), torch.flip(crop, dims=[-2])]
            prob_list.extend(torch.softmax(model(v), dim=1) for v in variants)
        probs = torch.stack(prob_list).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def save_submission(ids, probs, path):
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)}).sort_values('id')
    sub.to_csv(path, index=False)
    print('Saved:', path)
    print(sub['label'].value_counts().sort_index())
    return sub


## 7. Run 5-fold


In [7]:
def run_model_5fold(model_name):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    test_loader = make_loader(test_df, test_tfms, BATCH_SIZE, False, False)
    fold_probs, ids_reference, rows = [], None, []
    start_fold = START_FOLD.get(model_name, 1)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name = f'{model_name}_fold{fold}'
        probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
        ids_path = CHECKPOINT_DIR / f'ids_{fold_name}.npy'
        if fold < start_fold:
            if probs_path.exists() and ids_path.exists():
                probs = np.load(probs_path); ids = np.load(ids_path)
                fold_probs.append(probs); ids_reference = ids if ids_reference is None else ids_reference
                print(f'Fold {fold}: loaded {probs.shape}')
            continue

        print('\n' + '='*80)
        print(f'{model_name} fold {fold}/{N_SPLITS}')
        print('='*80)
        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)
        train_loader = make_loader(tr_df, train_tfms, BATCH_SIZE, True, True)
        val_loader = make_loader(va_df, val_tfms, BATCH_SIZE, False, True)

        model = build_model(model_name).to(DEVICE)
        hist, best_path, best_acc, best_epoch, elapsed = fit_model(model, fold_name, train_loader, val_loader)
        hist.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)

        ckpt = torch.load(best_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        probs, ids = predict_patch_tta(model, test_loader)
        np.save(probs_path, probs); np.save(ids_path, ids)
        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids)
        fold_probs.append(probs)
        save_submission(ids, probs, SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv')
        rows.append({'model': model_name, 'fold': fold, 'best_val_acc': best_acc,
                     'best_epoch': best_epoch, 'training_time_min': elapsed})
        del model
        torch.cuda.empty_cache()

    result_df = pd.DataFrame(rows)
    result_df.to_csv(OUTPUT_DIR / f'results_{model_name}.csv', index=False)
    display(result_df)
    if len(fold_probs) == N_SPLITS:
        mean_probs = np.mean(fold_probs, axis=0)
        np.save(CHECKPOINT_DIR / f'probs_{model_name}_5fold.npy', mean_probs)
        np.save(CHECKPOINT_DIR / f'ids_{model_name}_5fold.npy', ids_reference)
        save_submission(ids_reference, mean_probs, SUBMISSION_DIR / f'submission_{model_name}_5fold_tta_labels0.csv')
    else:
        print(f'{model_name}: {len(fold_probs)}/{N_SPLITS} folds disponibles')
    return result_df

all_results = []
for model_name in RUN_MODELS:
    all_results.append(run_model_5fold(model_name))
summary = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
summary.to_csv(OUTPUT_DIR / 'results_scratch_patch_heavy.csv', index=False)
display(summary)
if not summary.empty:
    display(summary.groupby('model')['best_val_acc'].agg(['count', 'mean', 'std', 'min', 'max']))



se_resnet34_patch_heavy fold 1/5


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

se_resnet34_patch_heavy_fold1 | ep 001 | train 0.1430/2.4942 | val 0.1966/2.1264 | best 0.1966 @ 1


  0%|          | 0/59 [00:00<?, ?it/s]

...

  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

se_resnet34_patch_heavy_fold1 | ep 018 | train 0.3792/1.6678 | val 0.3615/1.5897 | best 0.3975 @ 14


  0%|          | 0/59 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Ensemble scratch patch heavy


In [ ]:
probs_list, ids_reference, used = [], None, []
for model_name in RUN_MODELS:
    probs_path = CHECKPOINT_DIR / f'probs_{model_name}_5fold.npy'
    ids_path = CHECKPOINT_DIR / f'ids_{model_name}_5fold.npy'
    if probs_path.exists() and ids_path.exists():
        probs = np.load(probs_path)
        ids = np.load(ids_path)
        if ids_reference is None:
            ids_reference = ids
        else:
            assert np.array_equal(ids_reference, ids)
        probs_list.append(probs)
        used.append(model_name)

if probs_list:
    ensemble_probs = np.mean(probs_list, axis=0)
    np.save(CHECKPOINT_DIR / 'probs_scratch_patch_heavy_ensemble.npy', ensemble_probs)
    save_submission(ids_reference, ensemble_probs,
                    SUBMISSION_DIR / 'submission_scratch_patch_heavy_ensemble_tta_labels0.csv')
    print('Used:', used)
else:
    print('Aucun modele 5-fold complet pour ensemble.')
